# 10 — Silver Contracts & Processes

**Objetivo:** construir la primera representación canónica Silver de contratos y procesos, preservando lineage, segregando estado mutable y eliminando la explosión de joins observada en Bronze.

Este notebook **no crea features ML ni labels**. La lógica reusable vive en `src/argos/silver/`; aquí se orquesta, inspecciona evidencia y se valida el resultado.


## Invariantes heredados de M1

- `contracts.id_contrato` es único y no nulo.
- `contracts.proceso_de_compra -> processes.id_del_portafolio` es la relación empíricamente aceptada.
- `processes.id_del_portafolio` no es único en Bronze: existen múltiples payloads/observaciones por identidad.
- Los campos contractuales mutables/post-`t0` deben quedar físicamente separados.
- Silver **no equivale** a una whitelist de features; Gold volverá a exigir point-in-time correctness.


In [9]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files
from argos.config.silver import load_silver_spec
from argos.evaluation.m1_gate import evaluate_m1_gate
from argos.silver.contracts_processes import BUILD_IMPLEMENTATION

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

gate_path = paths.docs / "decisions" / "M1_GATE.md"
if gate_path.exists():
    gate_closed = "M1 ✅ CLOSED" in gate_path.read_text(encoding="utf-8")
else:
    _, gate_decision = evaluate_m1_gate(paths.reports, PROJECT_CONFIG)
    gate_closed = gate_decision.closed
assert gate_closed, "M1 no está formalmente cerrado; no iniciar Silver."

DATASETS, _ = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)
SILVER_SPEC = load_silver_spec(paths.configs / "silver.toml")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 160)

{
    "contracts": FILES["contracts"].name,
    "processes": FILES["processes"].name,
    "silver_schema_version": SILVER_SPEC.schema_version,
    "build_implementation": BUILD_IMPLEMENTATION,
    "duckdb_threads": SILVER_SPEC.runtime.duckdb_threads,
    "duckdb_memory_limit": SILVER_SPEC.runtime.duckdb_memory_limit,
    "m1_gate_source": str(gate_path) if gate_path.exists() else "re-evaluated from M1 reports",
}


{'contracts': 'SECOP_II_-_Contratos_Electrónicos_20260921.csv',
 'processes': 'SECOP_II_-_Procesos_de_Contratación_20260921.csv',
 'silver_schema_version': 'm2-silver-v1',
 'build_implementation': 'm2-10-out-of-core-v3',
 'duckdb_threads': 2,
 'duckdb_memory_limit': '8GB',
 'm1_gate_source': '/Users/macm1pro/Desktop/projects/argos/argos-code/docs/decisions/M1_GATE.md'}

## 1. Plan de routing de columnas

Antes de escribir Parquet inspeccionamos qué columnas irán a `contracts.parquet`, cuáles se segregan en `contracts_snapshot_state.parquet` y cuáles se excluyen del Silver analítico general por sensibilidad.


In [10]:
from argos.silver.contracts_processes import plan_contracts_processes_silver

routing = plan_contracts_processes_silver(
    FILES["contracts"],
    FILES["processes"],
    column_profile_path=paths.reports / "profiling" / "column_profile.parquet",
    silver_spec=SILVER_SPEC,
    sample_size=PROJECT_CONFIG["profiling"]["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

routing.groupby(["dataset", "route"]).size().rename("columns").reset_index()


,dataset,route,columns
0,contracts,core,63
1,contracts,excluded_sensitive,9
2,contracts,snapshot_state,13
3,processes,version,59


In [11]:
routing.loc[
    (routing["dataset"] == "contracts")
    & (routing["route"].isin(["snapshot_state", "excluded_sensitive"])),
    [
        "raw_column",
        "canonical_column",
        "semantic_type",
        "leakage_risk_hint",
        "route",
        "reason",
    ],
].sort_values(["route", "canonical_column"])


,raw_column,canonical_column,semantic_type,leakage_risk_hint,route,reason
53,Identificación Representante Legal,identificacion_representante_legal,pii,review,excluded_sensitive,semantic_type=pii
70,Número de cuenta,numero_de_cuenta,pii,review,excluded_sensitive,semantic_type=pii
81,Número de documento Ordenador de Pago,numero_de_documento_ordenador_de_pago,pii,review,excluded_sensitive,semantic_type=pii
75,Número de documento Ordenador del gasto,numero_de_documento_ordenador_del_gasto,pii,review,excluded_sensitive,semantic_type=pii
78,Número de documento supervisor,numero_de_documento_supervisor,pii,review,excluded_sensitive,semantic_type=pii
80,Tipo de documento Ordenador de Pago,tipo_de_documento_ordenador_de_pago,pii,review,excluded_sensitive,semantic_type=pii
74,Tipo de documento Ordenador del gasto,tipo_de_documento_ordenador_del_gasto,pii,review,excluded_sensitive,semantic_type=pii
77,Tipo de documento supervisor,tipo_de_documento_supervisor,pii,review,excluded_sensitive,semantic_type=pii
52,Tipo de Identificación Representante Legal,tipo_de_identificacion_representante_legal,pii,review,excluded_sensitive,semantic_type=pii
45,Dias adicionados,dias_adicionados,category_or_text,high,snapshot_state,M1 mutable/post-t0 leakage risk


## 2. Construcción Silver

La construcción produce cinco artefactos:

- `contracts.parquet`: contrato canónico, sin los campos de snapshot mutable conocidos.
- `contracts_snapshot_state.parquet`: estado actual/post-`t0` segregado.
- `process_versions.parquet`: payloads distintos de procesos después de deduplicar filas Bronze exactamente iguales.
- `processes.parquet`: identidad canónica `1 fila = 1 process_id` + diagnósticos de variantes.
- `contract_process_links.parquet`: `1 fila = 1 contrato`, sin expansión del join.


### Ejecución out-of-core

`processes` se deduplica mediante staging Parquet + índice estrecho de fingerprints; no se ejecuta un `GROUP BY ALL` ancho sobre el CSV completo. La configuración de Silver limita DuckDB a 2 threads y 8 GB para dejar headroom al kernel de Jupyter y permitir spill a `.cache/duckdb`.


**Implementación esperada:** `m2-10-out-of-core-v2`. Si la primera celda muestra otra implementación o no muestra `duckdb_memory_limit = 8GB`, el kernel está cargando una versión anterior del paquete.


In [4]:
from argos.silver.contracts_processes import build_contracts_processes_silver

result = build_contracts_processes_silver(
    FILES["contracts"],
    FILES["processes"],
    contracts_spec=DATASETS["contracts"],
    processes_spec=DATASETS["processes"],
    column_profile_path=paths.reports / "profiling" / "column_profile.parquet",
    reports_root=paths.reports,
    silver_dir=paths.silver,
    manifest_dir=paths.root / "artifacts" / "manifests",
    silver_spec=SILVER_SPEC,
    date_order=PROJECT_CONFIG["temporal"]["date_order"],
    sample_size=PROJECT_CONFIG["profiling"]["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
    overwrite=True,
)

{name: str(path.relative_to(paths.root)) for name, path in result.artifacts.items()}


[10] processes: staging raw payloads (streaming CSV -> Parquet)...
[10] processes: finding duplicate fingerprints (narrow aggregate)...
[10] processes: resolving candidate duplicates by exact raw equality...
[10] processes: materializing typed unique payloads...


{'contracts': 'data/silver/contracts.parquet',
 'contracts_snapshot_state': 'data/silver/contracts_snapshot_state.parquet',
 'process_versions': 'data/silver/process_versions.parquet',
 'processes': 'data/silver/processes.parquet',
 'contract_process_links': 'data/silver/contract_process_links.parquet'}

## 3. Auditoría de conversión de tipos

Para fechas y dinero Silver conserva tres representaciones: valor tipado, `__raw` y `__parse_status`. Los valores no reconocidos no desaparecen silenciosamente.


In [5]:
result.conversion_audit.sort_values(
    ["unparsed_rows", "dataset", "canonical_column"],
    ascending=[False, True, True],
)


,dataset,artifact,canonical_column,semantic_type,represented_rows,missing_rows,parsed_rows,unparsed_rows,parse_success_rate_nonmissing
14,contracts,contracts_snapshot_state,fecha_inicio_liquidacion,date,6066730,5404550,0,662180,0.000000
15,contracts,contracts_snapshot_state,fecha_fin_liquidacion,date,6066730,5404596,0,662134,0.000000
16,contracts,contracts_snapshot_state,fecha_de_notificacion_de_prorrogacion,date,6066730,5485610,0,581120,0.000000
3,contracts,contracts,valor_del_contrato,money,6066730,0,6066702,28,0.999995
2,contracts,contracts,fecha_de_fin_del_contrato,date,6066730,54613,6012117,0,1.000000
0,contracts,contracts,fecha_de_firma,date,6066730,426452,5640278,0,1.000000
1,contracts,contracts,fecha_de_inicio_del_contrato,date,6066730,468911,5597819,0,1.000000
13,contracts,contracts,presupuesto_general_de_la_nacion_pgn,money,6066730,0,6066730,0,1.000000
11,contracts,contracts,saldo_cdp,money,6066730,0,6066730,0,1.000000
12,contracts,contracts,saldo_vigencia,money,6066730,0,6066730,0,1.000000


**Interpretación:** un `unparsed_rows > 0` no se oculta ni se imputa. Si corresponde a un campo de snapshot mutable/deferred, queda registrado con su valor `__raw` para resolución posterior. Para `fecha_de_firma` sí exigimos que la población parseada reproduzca la evidencia temporal de M1, porque define `t0`.


## 4. Validación del componente 10

Estos checks son **bloqueantes para continuar a `11_silver_suppliers_bidders.ipynb`**. No son todavía el gate completo de M2; ese rol pertenece a `15_m2_gate.ipynb`.


In [6]:
result.validation


,check_id,category,blocking,passed,observed,requirement
0,contracts_row_count,contracts,True,True,"{""m1_expected"": 6066730, ""rows"": 6066730}",Silver contracts preserves the M1 Bronze contract population
1,contracts_primary_key,contracts,True,True,"{""distinct_contract_id"": 6066730, ""null_contract_id"": 0, ""rows"": 6066730}",contract_id is non-null and unique
2,snapshot_state_alignment,contracts,True,True,"{""contract_rows"": 6066730, ""state_rows"": 6066730}",one snapshot-state row per contract
3,mutable_fields_segregated,leakage,True,True,[],no M1 high-leakage field is stored in contracts.parquet
4,sensitive_fields_excluded,privacy,True,True,[],configured sensitive semantic types are not in general Silver contracts
5,t0_preserved,temporal,True,True,"{""m1_parsed"": 5640278, ""m1_raw_nonnull"": 5640278, ""parse_success"": 1.0, ""silver_parsed"": 5640278}",Silver uses the same explicit t0 parser and preserves the M1 parsed population
6,process_versions_represent_source,processes,True,True,"{""m1_expected_source_rows"": 9204423, ""null_process_id"": 0, ""represented_source_rows"": 9204423, ""version_rows"": 9067585}",deduplicated versions account for every Bronze process row and preserve a non-null process identity
7,process_identity_key,processes,True,True,"{""distinct_process_id"": 8540484, ""m1_expected_distinct"": 8540484, ""null_process_id"": 0, ""rows"": 8540484}",processes.parquet is one row per non-null process_id
8,contract_process_link_grain,relationships,True,True,"{""distinct_contracts"": 6066730, ""rows"": 6066730}",one contract-process link row per contract; no join expansion
9,contract_process_link_coverage,relationships,True,True,"{""coverage"": 0.9754900910375112, ""matched"": 5918035, ""missing_key"": 0, ""unmatched"": 148695}",coverage >= 90.00% after process canonicalization


In [7]:
from argos.common.duckdb import connect
from argos.common.sql import sql_string

con = connect(
    temp_directory=paths.root / ".cache" / "duckdb",
    threads=SILVER_SPEC.runtime.duckdb_threads,
    memory_limit=SILVER_SPEC.runtime.duckdb_memory_limit,
)
try:
    link_status = con.execute(
        f"""
        SELECT link_status, COUNT(*) AS contracts
        FROM read_parquet({sql_string(result.artifacts['contract_process_links'])})
        GROUP BY link_status
        ORDER BY contracts DESC
        """
    ).fetchdf()

    process_variants = con.execute(
        f"""
        SELECT
            COUNT(*) AS processes,
            AVG(version_count) AS avg_versions,
            MAX(version_count) AS max_versions,
            COUNT(*) FILTER (WHERE version_count > 1) AS processes_with_multiple_versions
        FROM read_parquet({sql_string(result.artifacts['processes'])})
        """
    ).fetchdf()
finally:
    con.close()

display(link_status)
display(process_variants)


,link_status,contracts
0,matched,5918035
1,unmatched_process,148695


,processes,avg_versions,max_versions,processes_with_multiple_versions
0,8540484,1.061718,7420,181101


In [8]:
blocking = result.validation.loc[result.validation["blocking"]]
failed = blocking.loc[~blocking["passed"], "check_id"].tolist()

print("10 ✅ PASS" if result.ready_for_next_notebook else "10 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
if failed:
    print("Failed:", ", ".join(failed))

assert result.ready_for_next_notebook, (
    "Notebook 10 remains OPEN. Resolve the failed checks before notebook 11."
)


10 ✅ PASS
Blocking checks: 10/10
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/silver_contracts_processes.json


## Criterio de salida

`10` termina únicamente si los checks bloqueantes pasan y existen los cinco Parquet Silver más el manifest.

**Siguiente paso:** `11_silver_suppliers_bidders.ipynb`. La relación directa bidder→supplier permanece sin resolver y se cerrará explícitamente en `13_entity_resolution.ipynb`; no se fuerza un join artificial en este notebook.
